<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab8-Probability-Generating-Functions-with-SymPy.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 8: Probability Generating Functions with SymPy

**Study Session 8: Discrete Probability Generating Function** · Davidson Open University

**In this lab you will:**
- write the probability generating function (PGF) G(z) of a discrete random variable;
- read probabilities from the coefficients of G(z);
- find the mean and variance from G′(1) and G″(1);
- use PGFs for sums of independent random variables, and check results by simulation.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '8.1': {'h': 'fa3c9d68f28d0a8f', 'hint': 'There is no z⁰ term, because the die never shows 0.', 'ok': 'Check: G_die.subs(z, 1) should be 1.'},
    '8.2': {'h': '9f611b36a5238050', 'hint': 'E(X) = G′(1): sp.diff(G_die, z).subs(z, 1).', 'ok': 'The mean score of a fair die is 3.5.'},
    '8.3': {'h': '3201e8b93c7bc7c2', 'hint': 'Var(X) = G″(1) + G′(1) − [G′(1)]². Use sp.diff(G_die, z, 2) for the second derivative.', 'ok': 'Exact fractions make it easy to compare with a hand calculation.'},
    '8.4': {'h': 'cf10a0571d8e0bb5', 'hint': 'Expand first with sp.expand(...), then use .coeff(z, 3). Use sp.Rational(1, 2) for 1/2.', 'ok': 'This equals C(5, 3)/2⁵.'},
    '8.5': {'h': '4b1920404a6fd5a8', 'hint': 'Build Gg = p*z / (1 - q*z) with p = sp.Rational(1, 4), then differentiate and substitute z = 1.', 'ok': 'On average you need 1/p = 4 trials for the first success.'},
    '8.6': {'h': '0703e99e4220e259', 'hint': 'Compute g1 = sp.diff(G, z).subs(z, 1) and g2 = sp.diff(G, z, 2).subs(z, 1); return (G.subs(z, 1), g1, g2 + g1 - g1**2).', 'ok': 'Your toolkit works for any PGF, including the Poisson distribution.', 'args': [((__import__("sympy").Rational(1, 3) + __import__("sympy").Rational(2, 3) * __import__("sympy").Symbol("z")) ** 4,), (__import__("sympy").exp(2 * (__import__("sympy").Symbol("z") - 1)),), (sum(__import__("sympy").Rational(1, 6) * __import__("sympy").Symbol("z") ** k for k in range(1, 7)),)]},
    '8.7': {'h': '18c578362dd79c2b', 'hint': 'sp.expand(G_die ** 2).coeff(z, 7)', 'ok': '7 is the most likely total of two dice.'},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
The **probability generating function** of a random variable X that takes values 0, 1, 2, … is

G(z) = E(zˣ) = P(X = 0) + P(X = 1)z + P(X = 2)z² + …

so the **coefficient of zᵏ is P(X = k)**. Three facts from Study Session 8:
- G(1) = 1 (the probabilities add up to 1);
- E(X) = G′(1);
- Var(X) = G″(1) + G′(1) − [G′(1)]².

**SymPy** can expand, differentiate and substitute. Run the cell for the binomial distribution with n = 3 and p = 1/2 (three fair coin tosses).

In [ ]:
import sympy as sp
z = sp.symbols('z')
p = sp.Rational(1, 2)
q = 1 - p
G = (q + p * z) ** 3
print('G(z) =', sp.expand(G))
print('P(X = 2) = coefficient of z² =', sp.expand(G).coeff(z, 2))
print('G(1) =', G.subs(z, 1))
G1 = sp.diff(G, z).subs(z, 1)
G2 = sp.diff(G, z, 2).subs(z, 1)
print('E(X) = G′(1) =', G1, '   Var(X) =', G2 + G1 - G1 ** 2)

Compare with the binomial formulas E(X) = np = 3/2 and Var(X) = npq = 3/4. SymPy keeps **exact fractions** such as 3/2, which is better than decimals for checking your working.

## Part B – Practise (about 20 minutes)
**B1. A fair die.** X is the score on a fair six-sided die, so P(X = k) = 1/6 for k = 1, …, 6. Write its PGF. *Hint:* `sum(sp.Rational(1, 6) * z ** k for k in range(1, 7))`.

In [ ]:
G_die = ...
answer_8_1 = G_die

In [ ]:
check('8.1', answer_8_1)

In [ ]:
answer_8_2 = ...   # E(X) for the die, from G_die

In [ ]:
check('8.2', answer_8_2)

In [ ]:
answer_8_3 = ...   # Var(X) for the die

In [ ]:
check('8.3', answer_8_3)

**B4.** X is binomial with n = 5 and p = 1/2. Use its PGF (q + pz)⁵ to find P(X = 3), the coefficient of z³.

In [ ]:
answer_8_4 = ...

In [ ]:
check('8.4', answer_8_4)

**B5. Geometric distribution.** If X is the number of trials up to and including the first success, with success probability p, then G(z) = pz / (1 − qz). For p = 1/4, find E(X) = G′(1).

In [ ]:
answer_8_5 = ...

In [ ]:
check('8.5', answer_8_5)

## Part C – Build a tool (about 20 minutes)
### C1. A general PGF toolkit
Complete the function `pgf_summary(G)`. It takes any PGF G (a SymPy expression in z) and returns a tuple **(G(1), E(X), Var(X))**, using the three facts from Part A. The cell tests it on several distributions.

In [ ]:
def pgf_summary(G):
    # YOUR CODE: return (G(1), E(X), Var(X)) using sp.diff and .subs(z, 1)
    ...

print(pgf_summary((sp.Rational(1, 2) + sp.Rational(1, 2) * z) ** 3))

In [ ]:
check('8.6', pgf_summary)

### C2. Sums of independent random variables
If X and Y are independent, the PGF of X + Y is **G_X(z) × G_Y(z)**. So the PGF for the total of two dice is G_die². Find P(total = 7) from the coefficient of z⁷.

In [ ]:
answer_8_7 = ...

In [ ]:
check('8.7', answer_8_7)

**Check by simulation.** A computer can "roll" two dice many times. The fraction of 7s should be close to your answer. Run the cell a few times: the result changes a little each time, but stays close.

In [ ]:
import random
rolls = 100_000
sevens = sum(1 for _ in range(rolls) if random.randint(1, 6) + random.randint(1, 6) == 7)
print('Fraction of 7s in', rolls, 'rolls:', sevens / rolls)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 8.** Use the toolkit on your assignment's PGF (q + pz)ⁿ: check G(1) = 1, find E(X) and Var(X), and read off the probability the assignment asks for. Show your working by hand in the assignment as well.

In [ ]:
# D1: check your Assignment 8 working

**D2. Reflect.** Answer in one or two sentences each:
1. Why must G(1) always equal 1?
2. In C2, why does multiplying the PGFs give the PGF of the total?
3. Your simulation gave a slightly different answer each time. Why, and how could you make it more accurate?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 8 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.